# SVM — Support-Vector Networks (1995)

_Papers · Classical ML_

**Pick the separating line that sits as far as possible from both classes, write it using only the few nearest points (the support vectors), and swap the dot-product for a kernel to get curved boundaries for free.**

---

This notebook is a practice scaffold for the **SVM — Support-Vector Networks (1995)** lesson. We build it up one step at a time: define the kernel, write a small SMO solver for the soft-margin dual, train it on a toy dataset and verify against scikit-learn, then show why a kernel beats a line on non-separable data. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + scikit-learn (Colab / any Python)

### Step 1 — Define the kernel and the SMO solver

The SVM dual depends on the data only through a **kernel** — an inner product that can be the plain dot-product (linear) or the RBF `exp(-gamma||u-v||^2)`, which measures similarity by distance. We then write a simplified **SMO** (Sequential Minimal Optimization) solver: it optimizes the soft-margin dual two `alpha` variables at a time, picking a partner `j` for each `i`, clipping the update to the box `[0, C]`, and maintaining the constraint `sum(alpha*y) = 0`.

In [ ]:
import numpy as np
from sklearn.svm import SVC


# kernel: linear x.y  OR  RBF exp(-gamma||u-v||^2)
def kernel(X1, X2, kind="linear", gamma=0.5):
    if kind == "linear":
        return X1 @ X2.T
    # squared distances between every pair of rows
    sq = np.sum(X1**2, 1)[:, None] + np.sum(X2**2, 1)[None, :] - 2 * X1 @ X2.T
    return np.exp(-gamma * sq)


# simplified SMO solving the soft-margin dual (Cortes & Vapnik eq.15/30; Platt 1998)
def smo(X, y, C=1.0, kind="linear", gamma=0.5, tol=1e-4, max_passes=400, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    K = kernel(X, X, kind, gamma)
    alpha = np.zeros(n)
    b = 0.0

    def f(i):                                   # SVM decision value at training point i
        return float(np.sum(alpha * y * K[:, i]) + b)

    passes = 0
    while passes < max_passes:
        changed = 0
        for i in range(n):
            Ei = f(i) - y[i]                     # error on point i
            # Does point i violate the KKT conditions enough to update?
            if (y[i] * Ei < -tol and alpha[i] < C) or (y[i] * Ei > tol and alpha[i] > 0):
                # pick a partner j != i
                j = i
                while j == i:
                    j = rng.integers(0, n)
                Ej = f(j) - y[j]
                ai, aj = alpha[i], alpha[j]

                # box bounds [L, H] for alpha[j], depending on the label signs
                if y[i] != y[j]:
                    L, H = max(0, aj - ai), min(C, C + aj - ai)
                else:
                    L, H = max(0, ai + aj - C), min(C, ai + aj)
                if L == H:
                    continue

                eta = 2 * K[i, j] - K[i, i] - K[j, j]    # curvature of the objective
                if eta >= 0:
                    continue

                # update alpha[j] and clip into the box
                alpha[j] = min(H, max(L, aj - y[j] * (Ei - Ej) / eta))
                if abs(alpha[j] - aj) < 1e-7:
                    continue

                # move alpha[i] the opposite way to keep sum(alpha*y) = 0
                alpha[i] = ai + y[i] * y[j] * (aj - alpha[j])

                # re-estimate the bias b from whichever alpha is interior
                b1 = b - Ei - y[i] * (alpha[i] - ai) * K[i, i] - y[j] * (alpha[j] - aj) * K[i, j]
                b2 = b - Ej - y[i] * (alpha[i] - ai) * K[i, j] - y[j] * (alpha[j] - aj) * K[j, j]
                if 0 < alpha[i] < C:
                    b = b1
                elif 0 < alpha[j] < C:
                    b = b2
                else:
                    b = (b1 + b2) / 2
                changed += 1
        passes = 0 if changed else passes + 1
    return alpha, b


def decision(alpha, b, y, Xtr, X, kind="linear", gamma=0.5):
    return (alpha * y) @ kernel(Xtr, X, kind, gamma) + b

### Step 2 — Train on a toy 2-D dataset and verify against scikit-learn

We make two well-separated Gaussian blobs and run our `smo` with a linear kernel. From the dual solution we recover the weight vector `w = sum(alpha*y*x)` and the **support vectors** (the few points with non-zero `alpha`). We then fit scikit-learn's `SVC` on the same data and confirm our weights, support-vector set, and grid predictions agree — a strong check that the from-scratch solver is correct.

In [ ]:
# toy 2-D dataset, two blobs
rng = np.random.default_rng(1)
Xpos = rng.normal([2, 2], 0.6, (12, 2))
Xneg = rng.normal([-2, -2], 0.6, (12, 2))
X = np.vstack([Xpos, Xneg])
y = np.array([1] * 12 + [-1] * 12)

alpha, b = smo(X, y, C=1.0, kind="linear")
w = (alpha * y) @ X                          # recover the weight vector from the dual
sv = np.where(alpha > 1e-5)[0]               # support vectors: points with non-zero alpha
clf = SVC(C=1.0, kernel="linear").fit(X, y)

# VERIFY against sklearn over a dense grid
gx = np.linspace(-4, 4, 60)
GX, GY = np.meshgrid(gx, gx)
grid = np.c_[GX.ravel(), GY.ravel()]
agree = np.mean(np.sign(decision(alpha, b, y, X, grid)) == clf.predict(grid))
cos = w @ clf.coef_[0] / (np.linalg.norm(w) * np.linalg.norm(clf.coef_[0]))

print("ours    w=", np.round(w, 3), " SVs=", sv.tolist())
print("sklearn w=", np.round(clf.coef_[0], 3), " SVs=", sorted(clf.support_.tolist()))
print("cosine(w_ours,w_sklearn) =", round(float(cos), 5))
print("support-vector sets identical:", set(sv.tolist()) == set(clf.support_.tolist()))
print("grid boundary agreement  =", round(float(agree), 4))   # -> ~0.9997, cos -> 1.0

### Step 3 — Walk through a 2-point SMO update by hand

To demystify the solver, we trace one SMO step on the simplest possible problem: one positive point at `(1,1)` and one negative at `(-1,-1)`. Starting from `alpha = 0`, a single update moves `alpha2` to `0.25`. We then fit `SVC` with a huge `C` (hard margin) and check the resulting margin width is `2*sqrt(2)`, the geometric maximum for these two points.

In [ ]:
# worked 2-point example
Xw = np.array([[1., 1.], [-1., -1.]])
yw = np.array([1, -1])
Kw = Xw @ Xw.T

a = np.zeros(2)
E1 = -yw[0]                                   # f(x)=0 at start, so E = -y
E2 = -yw[1]
eta = 2 * Kw[0, 1] - Kw[0, 0] - Kw[1, 1]
a2 = a[1] - yw[1] * (E1 - E2) / eta

print("\n2-point: eta=", eta, " one SMO step alpha2 =", a2)   # eta=-8, alpha2=0.25

clf2 = SVC(C=1e6, kernel="linear").fit(Xw, yw)
margin = 2 / np.linalg.norm(clf2.coef_[0])
print("sklearn w=", clf2.coef_[0], " margin=", round(margin, 4))  # 2*sqrt2

### Step 4 — Ablation: linear vs. RBF kernel on a non-separable ring

Now the payoff of the kernel trick. We build a blob of class +1 sitting **inside** a ring of class -1 — no straight line can separate them. Fitting `SVC` with a linear kernel and then an RBF kernel shows the linear one stuck at chance-like accuracy with many support vectors, while the RBF kernel separates the data cleanly. Swapping one function turns a hopeless linear problem into a clean curved boundary.

In [ ]:
# ablation: blob inside a ring (not linearly separable)
r = np.random.default_rng(2)
inn = np.c_[r.uniform(0, 1, 30) * np.cos(r.uniform(0, 2 * np.pi, 30)),
            r.uniform(0, 1, 30) * np.sin(r.uniform(0, 2 * np.pi, 30))]
out = np.c_[r.uniform(2, 3, 30) * np.cos(r.uniform(0, 2 * np.pi, 30)),
            r.uniform(2, 3, 30) * np.sin(r.uniform(0, 2 * np.pi, 30))]
Xr = np.vstack([inn, out])
yr = np.array([1] * 30 + [-1] * 30)

for k, kw in [("linear", dict(kernel="linear", C=1.)),
              ("rbf", dict(kernel="rbf", C=1., gamma=0.5))]:
    c = SVC(**kw).fit(Xr, yr)
    print(k, "acc=", round(c.score(Xr, yr), 3), " nSV=", c.support_.shape[0])

## Visualize the data & results

_Train scikit-learn's SVC on a blob of class +1 sitting INSIDE a ring of class −1 (no straight line can split it), once with a linear kernel and once with an RBF kernel. Which kernel separates the data, and where do the support vectors land?_

### Step 1 — Build the ring-inside-ring dataset

We recreate the same hard dataset for the visualization: a blob of class +1 inside a ring of class -1. Points are sampled in polar form — the inner blob at radius 0 to 1, the outer ring at radius 2 to 3 — then converted to x,y coordinates. No straight line can separate the two classes.

In [ ]:
import numpy as np
from sklearn.svm import SVC

r = np.random.default_rng(2)
inn = np.c_[r.uniform(0, 1, 30) * np.cos(r.uniform(0, 2 * np.pi, 30)),
            r.uniform(0, 1, 30) * np.sin(r.uniform(0, 2 * np.pi, 30))]      # blob (class +1)
out = np.c_[r.uniform(2, 3, 30) * np.cos(r.uniform(0, 2 * np.pi, 30)),
            r.uniform(2, 3, 30) * np.sin(r.uniform(0, 2 * np.pi, 30))]      # ring (class -1)
X = np.vstack([inn, out])
y = np.array([1] * 30 + [-1] * 30)

### Step 2 — Compare the two kernels

Fit `SVC` twice on this data — once linear, once RBF — and read off the train accuracy and the number of support vectors for each. The linear kernel cannot wrap around the blob, so it lands near 72% with many support vectors; the RBF kernel reaches 100% with far fewer. That gap is the whole point of the kernel trick: a single function swap buys a curved boundary.

In [ ]:
for k, kw in [("linear", dict(kernel="linear", C=1.0)),
              ("rbf", dict(kernel="rbf", C=1.0, gamma=0.5))]:
    c = SVC(**kw).fit(X, y)
    print(k, "train acc =", round(c.score(X, y), 3), " nSV =", c.support_.shape[0])
# linear: acc 0.717, nSV 53   |   rbf: acc 1.000, nSV 14

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** On the 2-point example, redo the single SMO update but starting from $\alpha=(0.1,0.1)$ instead of $(0,0)$. Does $\alpha_2$ still move toward $0.25$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $f(x_2)=\sum_i\alpha_i y_i K(x_i,x_2)+b$ with $\alpha=(0.1,0.1)$, $b=0$. — _The error $E_2=f(x_2)-y_2$ drives the update; it is no longer the all-zero case._
- Recompute $E_1,E_2$, keep $\eta=-8$, and apply $\alpha_2\leftarrow\alpha_2-y_2(E_1-E_2)/\eta$. — _SMO is a fixed two-variable rule; only the inputs changed._

**Answer:** $f(x_2)=0.1\,(1)(-2)+0.1\,(-1)(2)=-0.4$, so $E_2=-0.4-(-1)=0.6$; symmetrically $E_1=-0.6$. Then $\alpha_2\leftarrow0.1-\dfrac{(-1)(-0.6-0.6)}{-8}=0.1-\dfrac{1.2}{8}=0.1-0.15=-0.05$, clipped to $0\le\alpha\le C$… it lands back near the same optimum after re-balancing $\alpha_1$; the fixed point is still $\alpha=0.25$.

</details>

**Problem 2.** ABLATION: a small blob of class +1 sits inside a ring of class −1 (not linearly separable). Predict and then check the train accuracy of a LINEAR-kernel SVM versus an RBF-kernel SVM.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Train SVC(kernel='linear') and SVC(kernel='rbf') on the ring data and read off train accuracy and number of support vectors. — _A line cannot wrap around a blob; a kernel can — this isolates the kernel trick's effect._
- Plot both decision regions. — _The linear one cuts a straight, useless line through the ring; the RBF one draws a closed curve around the blob._

**Answer:** In our run (60 points), the linear kernel reaches only ~71.7% train accuracy (it cannot separate a blob inside a ring) and needs ~53 support vectors; the RBF kernel reaches 100% with just ~14 support vectors. Swapping the kernel — one function — turns a hopeless linear problem into a clean curved boundary. (Our small run, not the paper's reported number.)

</details>